In [ ]:
import time, os
import requests as http_requests

NOTEBOOK_NAME = "anime-omni-tts"
STEP_NAME = "step_omni"

# Carregar secrets
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _ks(n):
        try: return _s.get_secret(n)
        except: return ""
    DATABASE_URL = _ks("DATABASE_URL")
    PROJECT_ID = _ks("PIPELINE_PROJECT_ID")
    PIPELINE_WEBHOOK_URL = _ks("PIPELINE_WEBHOOK_URL")
    TASK_KEY = _ks("PIPELINE_TASK_KEY")
except:
    DATABASE_URL = os.getenv("DATABASE_URL", "")
    PROJECT_ID = os.getenv("PIPELINE_PROJECT_ID", "")
    PIPELINE_WEBHOOK_URL = os.getenv("PIPELINE_WEBHOOK_URL", "")
    TASK_KEY = os.getenv("PIPELINE_TASK_KEY", os.getenv("TASK_KEY", ""))

if not TASK_KEY:
    TASK_KEY = "omni-tts-pt3"

# Extrair numero da parte (ex: "omni-tts-pt2" -> 2)
try:
    TTS_PART_NUM = int(TASK_KEY.split("-pt")[-1])
except:
    TTS_PART_NUM = 1

print(f"[TTS] TASK_KEY={TASK_KEY} | Parte={TTS_PART_NUM}")

_cell_timers = {}

def _db_exec(query, params):
    if not DATABASE_URL: return
    try:
        import psycopg2
        conn = psycopg2.connect(DATABASE_URL)
        cur = conn.cursor()
        cur.execute(query, params)
        conn.commit()
        cur.close()
        conn.close()
    except: pass

def cell_start(idx, name=""):
    _cell_timers[idx] = time.time()
    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")
    if not PROJECT_ID: return
    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,'running',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))
    _db_exec("UPDATE pipeline_cell_tracking SET status='running',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))

def cell_end(idx, status="done", msg=""):
    elapsed = ""
    if idx in _cell_timers:
        s = int(time.time() - _cell_timers.pop(idx))
        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"
    icon = "OK" if status == "done" else "ERRO"
    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'_'*50}")
    if not PROJECT_ID: return
    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))

def report_step(status, msg=""):
    if PROJECT_ID and PIPELINE_WEBHOOK_URL:
        try:
            http_requests.post(f"{PIPELINE_WEBHOOK_URL}/webhook/status", json={"project_id": PROJECT_ID, "step": STEP_NAME, "status": status, "message": msg}, timeout=15)
        except: pass
    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")
    if not PROJECT_ID: return
    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))
    _db_exec("INSERT INTO pipeline_logs (project_id,step,status,message) VALUES (%s::uuid,%s,%s,%s)", (PROJECT_ID, STEP_NAME, status, msg))

def report_tts_part_done(part_num, msg=""):
    """Reporta via webhook que uma parte TTS especifica foi concluida."""
    step_name = f"step_omni_tts_pt{part_num}"
    if PROJECT_ID and PIPELINE_WEBHOOK_URL:
        try:
            http_requests.post(f"{PIPELINE_WEBHOOK_URL}/webhook/status", json={
                "project_id": PROJECT_ID, "step": step_name,
                "status": "done", "message": msg
            }, timeout=15)
        except: pass
    print(f"\n[TTS PT{part_num}] Reportado como done via webhook.")

def fetch_project_opts():
    if not DATABASE_URL or not PROJECT_ID: return {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}
    try:
        import psycopg2
        from psycopg2.extras import RealDictCursor
        conn = psycopg2.connect(DATABASE_URL, cursor_factory=RealDictCursor)
        cur = conn.cursor()
        cur.execute("SELECT bg_audio, srt_type, azure_enabled FROM pipeline_projects WHERE id=%s::uuid", (PROJECT_ID,))
        row = cur.fetchone()
        cur.close()
        conn.close()
        return dict(row) if row else {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}
    except Exception as e:
        print(f"Erro ao buscar opts: {e}")
        return {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}

PROJECT_OPTS = fetch_project_opts()
AZURE_ENABLED = PROJECT_OPTS.get("azure_enabled", True)
print("Tracking configurado!", PROJECT_OPTS)


In [ ]:
cell_start(1, "Setup TTS: Drive + Clonagem + Modelos")
# Setup simplificado para notebooks omni-tts-ptX.
# Carrega: Drive, audio de referencia, clientes Gemini/Azure/OpenAI.
# NAO carrega: Whisper, FunASR, Demucs, Pyannote (esses ficam no assemble).

import os, sys, shutil, time, json, nest_asyncio, gc
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

print("Instalando dependencias...")
os.system("apt-get install -y ffmpeg > /dev/null 2>&1")
os.system("pip install python-dotenv google-genai google-auth google-auth-httplib2 google-api-python-client omnivoice torchaudio pydub openai nest_asyncio stable-ts > /dev/null 2>&1")

import torch
from google import genai as google_genai_client
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from openai import OpenAI
from pydub import AudioSegment
from dotenv import load_dotenv
nest_asyncio.apply()

# Configuracoes de velocidade
TARGET_MIN_SPEED = 1.1
TARGET_MAX_SPEED = 1.35
MAX_AUDIO_RETRIES = 4
MAX_TEXT_RETRIES  = 3

NOME_NARRADOR = "Alessandro"
BATCH_SIZE    = 50

# Modelos Gemini (reescrita de texto)
MODELS_REESCRITA = [
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview"
]
OPENAI_FALLBACK_MODEL = "gpt-5.4"
USE_OPENAI_FALLBACK   = True
AZURE_ENABLED = True  # sobrescrito por PROJECT_OPTS abaixo

# Caminhos
BASE_PATH            = "/kaggle/working/AUDIO_DUB"
TEMP_PATH            = "/kaggle/working/temp_audio"
ANIME_AUDIO_ORIGINAL = "/kaggle/working/input/anime_audio.mp3"
MODEL_OMNIVOICE_PATH = "k2-fsa/OmniVoice"

os.chdir("/kaggle/working")
for p in [BASE_PATH, TEMP_PATH, "/kaggle/working/input"]:
    os.makedirs(p, exist_ok=True)

# Limpeza de temporarios
for f in os.listdir("/kaggle/working"):
    if f.endswith((".mp3", ".wav", ".srt")):
        try: os.remove(f"/kaggle/working/{f}")
        except: pass

# Carregar chaves
def _load_credentials():
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        def _get(name):
            try: return _s.get_secret(name)
            except: return ""
        print("Carregando chaves do Kaggle Secrets...")
        return _get
    except ImportError:
        pass
    load_dotenv()
    print("Carregando chaves do .env (local)...")
    return lambda name: os.getenv(name, "")

_get_secret = _load_credentials()
GEMINI_API_KEY          = _get_secret("GEMINI_API_KEY")
OPENAI_API_KEY          = _get_secret("OPENAI_API_KEY")
DRIVE_ACCESS_TOKEN      = _get_secret("DRIVE_ACCESS_TOKEN")
DRIVE_REFRESH_TOKEN     = _get_secret("DRIVE_REFRESH_TOKEN")
DRIVE_CLIENT_ID         = _get_secret("DRIVE_CLIENT_ID")
DRIVE_CLIENT_SECRET     = _get_secret("DRIVE_CLIENT_SECRET")
AZURE_OPENAI_ENDPOINT   = _get_secret("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY    = _get_secret("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = _get_secret("AZURE_OPENAI_DEPLOYMENT")

for _name, _val in [("GEMINI_API_KEY", GEMINI_API_KEY), ("DRIVE_REFRESH_TOKEN", DRIVE_REFRESH_TOKEN)]:
    if not _val:
        print(f"  ATENCAO: '{_name}' nao encontrada!")
    else:
        print(f"  [{_name}] OK")

# Google Drive
print("Autenticando Google Drive...")
try:
    creds = Credentials(
        token=DRIVE_ACCESS_TOKEN if DRIVE_ACCESS_TOKEN else None,
        refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=DRIVE_CLIENT_ID,
        client_secret=DRIVE_CLIENT_SECRET,
        scopes=["https://www.googleapis.com/auth/drive"]
    )
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=creds)
    print("Google Drive autenticado!")
except Exception as e:
    drive_service = None
    print(f"Falha na autenticacao do Drive: {e}")

def _buscar_id(caminho_no_drive):
    partes = caminho_no_drive.strip("/").split("/")
    parent_id = "root"
    for parte in partes:
        query = f"name='{parte}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id, mimeType)").execute()
        arquivos = results.get("files", [])
        if not arquivos: return None
        parent_id = arquivos[0]["id"]
    return parent_id

def _garantir_pasta(caminho_pasta):
    partes = caminho_pasta.strip("/").split("/")
    parent_id = "root"
    for pasta in partes:
        query = f"name='{pasta}' and '{parent_id}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        if existentes:
            parent_id = existentes[0]["id"]
        else:
            nova = drive_service.files().create(
                body={"name": pasta, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
                fields="id"
            ).execute()
            parent_id = nova["id"]
    return parent_id

def baixar_do_drive(caminho_no_drive, destino_local):
    if os.path.exists(destino_local): return True
    try:
        file_id = _buscar_id(caminho_no_drive)
        if not file_id: return False
        request = drive_service.files().get_media(fileId=file_id)
        with open(destino_local, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()
        return True
    except: return False

def salvar_no_drive(caminho_local, caminho_destino_drive):
    if drive_service is None or not os.path.exists(caminho_local): return
    try:
        partes = caminho_destino_drive.strip("/").split("/")
        nome_arquivo = partes[-1]
        pasta_drive  = "/".join(partes[:-1]) if len(partes) > 1 else ""
        parent_id = _garantir_pasta(pasta_drive) if pasta_drive else "root"
        query = f"name='{nome_arquivo}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        media = MediaFileUpload(caminho_local, resumable=True)
        if existentes:
            drive_service.files().update(fileId=existentes[0]["id"], media_body=media).execute()
        else:
            drive_service.files().create(
                body={"name": nome_arquivo, "parents": [parent_id]},
                media_body=media, fields="id"
            ).execute()
        print(f"  Salvo no Drive: {caminho_destino_drive}")
    except Exception as e:
        print(f"  Erro ao salvar {caminho_destino_drive}: {e}")

# Baixar arquivos do Drive
if drive_service:
    print("\nBaixando arquivos do Drive...")
    baixar_do_drive("KAGGLE/AUDIO_DUB/INPUT/anime_audio.mp3", ANIME_AUDIO_ORIGINAL)
    baixar_do_drive(f"KAGGLE/AUDIO_DUB/roteiro_pt{TTS_PART_NUM}.json", f"{BASE_PATH}/roteiro_pt{TTS_PART_NUM}.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/identificacao_anime.json", f"{BASE_PATH}/identificacao_anime.json")
    if TTS_PART_NUM == 1:
        baixar_do_drive("KAGGLE/AUDIO_DUB/intro_info.json", f"{BASE_PATH}/intro_info.json")

# Carregar identificacao do anime
try:
    with open(f"{BASE_PATH}/identificacao_anime.json", 'r', encoding='utf-8') as f:
        _anime_info = json.load(f)
    selected_anime = _anime_info.get("title", "Desconhecido")
    protagonist    = _anime_info.get("protagonist", "Narrador")
    anime_info     = _anime_info
except:
    selected_anime, protagonist, anime_info = "Desconhecido", "Narrador", {}

print(f"Anime: {selected_anime} | Protagonista: {protagonist}")

# Audio de referencia para clonagem
REF_AUDIO_PATH = ""
REF_TEXT       = ""

if drive_service and NOME_NARRADOR:
    from difflib import SequenceMatcher
    def _sim(a, b): return SequenceMatcher(None, a.lower(), b.lower()).ratio()
    _clonagem_drive = "KAGGLE/AUDIO_DUB/INPUT/CLONAGEM"
    _clonagem_local = f"{BASE_PATH}/INPUT/CLONAGEM"
    os.makedirs(_clonagem_local, exist_ok=True)
    _pid = _buscar_id(_clonagem_drive)
    if _pid:
        _res = drive_service.files().list(
            q=f"'{_pid}' in parents and trashed=false and mimeType contains 'audio/'",
            fields="files(id, name)"
        ).execute()
        _melhor, _score = None, 0.0
        for _arq in _res.get('files', []):
            _s = _sim(NOME_NARRADOR, os.path.splitext(_arq['name'])[0])
            if NOME_NARRADOR.lower() in _arq['name'].lower(): _s = max(_s, 0.85)
            if _s > _score: _score, _melhor = _s, _arq
        if _melhor and _score > 0.6:
            _local = f"{_clonagem_local}/{_melhor['name']}"
            if not os.path.exists(_local):
                _req = drive_service.files().get_media(fileId=_melhor['id'])
                with open(_local, "wb") as _f:
                    _dl = MediaIoBaseDownload(_f, _req)
                    _done = False
                    while not _done: _, _done = _dl.next_chunk()
            REF_AUDIO_PATH = _local
            if _local.endswith('.m4a'):
                _ref_wav = _local.replace('.m4a', '.wav')
                if not os.path.exists(_ref_wav):
                    AudioSegment.from_file(_local, format='m4a').export(_ref_wav, format='wav')
                REF_AUDIO_PATH = _ref_wav
            print(f"  Referencia de clonagem: {REF_AUDIO_PATH}")

# Clientes de API
client_gemini = google_genai_client.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
client_openai = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-") else None
client_azure  = None
if AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT:
    client_azure = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY)

QUOTA_ERROR_CODES = {429, 503, 500}
QUOTA_ERROR_MSGS  = ["quota", "rate limit", "resource exhausted", "overloaded", "too many requests", "503", "500", "internal", "timeout"]

def _is_quota_error(e):
    msg = str(e).lower()
    code = getattr(e, "code", getattr(e, "status_code", None))
    if code in QUOTA_ERROR_CODES: return True
    return any(m in msg for m in QUOTA_ERROR_MSGS)

def azure_generate(prompt, response_format=None, system_instruction=None):
    if not client_azure:
        raise RuntimeError("client_azure nao inicializado")
    messages = []
    if system_instruction:
        messages.append({"role": "system", "content": str(system_instruction)})
    messages.append({"role": "user", "content": str(prompt)})
    resp = client_azure.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT, messages=messages, temperature=1.0
    )
    return resp.choices[0].message.content

def gemini_generate(model_chain, contents, config=None, task_name=None):
    first_model = model_chain[0]
    try:
        kwargs = {"model": first_model, "contents": contents}
        if config: kwargs["config"] = config
        response = client_gemini.models.generate_content(**kwargs)
        print(f"  [OK] {task_name or 'gemini_generate'} -> {first_model}")
        return response
    except Exception as e:
        if AZURE_ENABLED and client_azure:
            try:
                sys_inst = config.get("system_instruction") if config else None
                resp_fmt = {"type": "json_object"} if config and config.get("response_mime_type") == "application/json" else None
                azure_text = azure_generate(contents, resp_fmt, sys_inst)
                print(f"  [OK] {task_name or 'gemini_generate'} -> Azure")
                class FakeResp:
                    def __init__(self, text): self.text = text
                return FakeResp(azure_text)
            except: pass
        last_exc = e
        for model in model_chain[1:]:
            try:
                kwargs = {"model": model, "contents": contents}
                if config: kwargs["config"] = config
                response = client_gemini.models.generate_content(**kwargs)
                print(f"  [OK] {task_name or 'gemini_generate'} -> {model}")
                return response
            except Exception as e2:
                last_exc = e2
                continue
        raise RuntimeError(f"Todos os modelos falharam. Ultimo erro: {last_exc}")

GPU_COUNT = torch.cuda.device_count()
device    = "cuda" if torch.cuda.is_available() else "cpu"
OMNIVOICE_PORTS = []  # preenchido pela cel8

AZURE_ENABLED = PROJECT_OPTS.get("azure_enabled", True)

print(f"""
=================================================================
  TTS SETUP | Parte {TTS_PART_NUM}
=================================================================
  Velocidade alvo : {TARGET_MIN_SPEED}x - {TARGET_MAX_SPEED}x
  GPUs            : {GPU_COUNT}
  Clonagem        : {REF_AUDIO_PATH or 'Nao disponivel'}
  Azure habilitado: {AZURE_ENABLED}
=================================================================
""")

cell_end(1, "done", f"Setup TTS pt{TTS_PART_NUM} concluido")


In [ ]:
cell_start(8, "Servidores OmniVoice")
# @title 🎙️ 1b. Servidores OmniVoice (GPU0 + GPU1)
# @markdown Sobe um servidor por GPU disponível. Rode antes da Cel6 (Fábrica de Áudio).
import subprocess, urllib.request, time, os, threading
import torch

if should_run("tts"):
    GPU_COUNT = torch.cuda.device_count()
    print(f"GPUs disponíveis: {GPU_COUNT}")
    print("Subindo servidores OmniVoice...")

    # Corrige path do modelo OmniVoice no CLI
    demo_file = "/usr/local/lib/python3.12/dist-packages/omnivoice/cli/demo.py"
    if os.path.exists(demo_file):
        with open(demo_file, "r") as f:
            content = f.read()
        if '"k2-fsa/OmniVoice"' in content:
            content = content.replace('"k2-fsa/OmniVoice"', f'"{MODEL_OMNIVOICE_PATH}"')
            with open(demo_file, "w") as f:
                f.write(content)
            print("CLI OmniVoice corrigido para dataset local.")

    _server_procs = []
    OMNIVOICE_PORTS = []

    for gpu_idx in range(min(GPU_COUNT, 2)):   # máximo 2 servidores
        port  = 8001 + gpu_idx
        log   = open(f"/kaggle/working/omnivoice_gpu{gpu_idx}.log", "w")
        env   = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_idx)}
        proc  = subprocess.Popen(
            ["omnivoice-demo", "--ip", "127.0.0.1", "--port", str(port)],
            env=env, stdout=log, stderr=subprocess.STDOUT
        )
        _server_procs.append(proc)
        OMNIVOICE_PORTS.append(port)
        print(f"  GPU{gpu_idx} → porta {port} iniciando...")

    def _aguardar_servidor(port, label):
        for i in range(40):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=2)
                print(f"  {label} online!")
                return True
            except:
                time.sleep(5)
                if (i + 1) % 4 == 0:
                    print(f"  [{(i+1)*5}s] {label} subindo...")
        print(f"  {label} nao respondeu. Verifique o log.")
        return False

    # Espera todos os servidores em paralelo
    threads = [
        threading.Thread(target=_aguardar_servidor, args=(port, f"GPU{idx}:porta{port}"))
        for idx, port in enumerate(OMNIVOICE_PORTS)
    ]
    for t in threads: t.start()
    for t in threads: t.join()

    print(f"\nServidores ativos: {OMNIVOICE_PORTS}")
    print("Pronto para dublar!")
    cell_end(8, "done", "Servidores OmniVoice ativos")
else:
    OMNIVOICE_PORTS = []
    print("TTS não é necessário para esta task. Ignorando inicialização do OmniVoice.")
    cell_end(8, "done", "OmniVoice ignorado")

In [ ]:
cell_start(9, "Fabrica de Audio")
# @title 🎙️ 6. Fábrica de Áudio: Dublagem (NARRACAO) + Recorte Original (CENA)
# @markdown Gera Video Completo com TTS clonado e recortes do anime.
import os
import re
import json
import soundfile as sf
from tqdm.notebook import tqdm
from pydub import AudioSegment
from pydub.silence import detect_leading_silence

# --- Lê configurações globais da Célula 1 ---
TARGET_MIN_SPEED  = globals().get('TARGET_MIN_SPEED', 1.0)
TARGET_MAX_SPEED  = globals().get('TARGET_MAX_SPEED', 1.25)
MAX_RETRIES       = globals().get('MAX_AUDIO_RETRIES', 4)
MAX_TEXT_RETRIES  = globals().get('MAX_TEXT_RETRIES', 3)
BASE_PATH         = globals().get('BASE_PATH', '/kaggle/working/AUDIO_DUB')
TEMP_DIR          = globals().get('TEMP_PATH', f'{BASE_PATH}/TEMP')
ANIME_AUDIO_ORIGINAL = globals().get('ANIME_AUDIO_ORIGINAL', '')
current_protagonist  = globals().get('protagonist', 'Narrador')

os.makedirs(TEMP_DIR, exist_ok=True)

print(f"Narrador: {current_protagonist} | 3a Pessoa")

# --- Voz de Clonagem ---
_ref_audio = globals().get('REF_AUDIO_PATH', '')
_ref_text  = globals().get('REF_TEXT', '')

if _ref_audio and _ref_audio.endswith('.m4a') and os.path.exists(_ref_audio):
    _ref_wav = _ref_audio.replace('.m4a', '.wav')
    if not os.path.exists(_ref_wav):
        AudioSegment.from_file(_ref_audio, format='m4a').export(_ref_wav, format='wav')
    _ref_audio = _ref_wav

if not _ref_audio or not os.path.exists(_ref_audio):
    print('Clonagem nao disponivel. Usando sintese padrao.')
    REF_AUDIO_PATH = ''
    REF_TEXT       = ''
else:
    REF_AUDIO_PATH = _ref_audio
    REF_TEXT       = _ref_text
    print(f'Clonagem ativa: {REF_AUDIO_PATH}')

# --- Áudio original do anime (para recortes CENA) ---
# Carregado aqui uma vez; reutilizado por recortar_cena() em ambos os modos.
anime_audio_full = None
if ANIME_AUDIO_ORIGINAL and os.path.exists(ANIME_AUDIO_ORIGINAL):
    anime_audio_full = AudioSegment.from_file(ANIME_AUDIO_ORIGINAL)
    print(f"Audio do anime carregado ({len(anime_audio_full)/1000:.1f}s)")
else:
    print("AVISO: Audio original nao encontrado. Blocos CENA ficarao silenciosos.")

# --- Clientes OmniVoice (um por GPU) ---
from gradio_client import Client, handle_file
import shutil, threading
from concurrent.futures import ThreadPoolExecutor

_portas = globals().get('OMNIVOICE_PORTS', []) or [8001]
omni_clients = []
for _p in _portas:
    try:
        omni_clients.append(Client(f"http://127.0.0.1:{_p}/"))
        print(f"OmniVoice conectado na porta {_p}")
    except Exception as _e:
        print(f"Porta {_p} offline: {_e}")

if not omni_clients:
    # fallback: tenta 8001
    try:
        omni_clients.append(Client("http://127.0.0.1:8001/"))
    except:
        print("AVISO: Nenhum servidor OmniVoice disponivel!")

_num_gpus = len(omni_clients)
print(f"  {_num_gpus} servidor(es) ativo(s) — TTS {'paralelo' if _num_gpus > 1 else 'sequencial'}")

# Lock para salvar progresso com segurança entre threads
_save_lock = threading.Lock()

# --- Funções de Áudio ---
def limpar_texto(texto):
    return re.sub(r'[*"(){}\[\]_~]', '', texto).strip()

def create_silent_audio(filename, duration_ms=1000):
    AudioSegment.silent(duration=duration_ms).export(filename, format='mp3')
    return duration_ms / 1000.0

def trim_audio_silence(wav_in, mp3_out, padding_ms=50):
    try:
        sound = AudioSegment.from_file(wav_in)
        st = detect_leading_silence(sound, -60.0, 10)
        en = detect_leading_silence(sound.reverse(), -60.0, 10)
        trimmed = sound[st:max(st + 100, len(sound) - en)]
        final = AudioSegment.silent(duration=padding_ms) + trimmed + AudioSegment.silent(duration=padding_ms)
        final.export(mp3_out, format='mp3')
        if os.path.exists(wav_in): os.remove(wav_in)
        return len(final) / 1000.0
    except:
        return 0.0

def generate_tts(text, filename, gpu_idx=0):
    texto = limpar_texto(text)
    if len(texto) < 2: return create_silent_audio(filename, 1000)
    # du = hint de duração para o OmniVoice. O loop de otimização corrige depois.
    est_duration = max(2.0, min(len(texto) / 13.0, 30.0))
    client = omni_clients[gpu_idx % len(omni_clients)] if omni_clients else None
    if not client:
        return create_silent_audio(filename, 1500)
    try:
        if REF_AUDIO_PATH and os.path.exists(REF_AUDIO_PATH):
            result = client.predict(
                text=texto, lang="Portuguese",
                ref_aud=handle_file(REF_AUDIO_PATH),
                ref_text=REF_TEXT if REF_TEXT else texto[:50],
                instruct="male, young adult",
                ns=32.0, gs=2.0, dn=True, sp=1.0, du=est_duration,
                pp=True, po=True, api_name="/_clone_fn"
            )
        else:
            result = client.predict(
                text=texto, lang="Portuguese",
                ns=32.0, gs=3.0, dn=True, sp=1.2, du=est_duration,
                pp=True, po=True,
                param_9="Male / 男", param_10="Young Adult / 青年",
                param_11="Moderate Pitch / 中音调",
                param_12="Auto", param_13="Auto", param_14="Auto",
                api_name="/_design_fn"
            )
        audio_path = result[0]
        if audio_path is None or not os.path.exists(audio_path):
            return create_silent_audio(filename, 1500)
        wav_local = filename.replace('.mp3', '.wav')
        shutil.copy(audio_path, wav_local)
        return trim_audio_silence(wav_local, filename)
    except Exception as e:
        print(f'Erro TTS (GPU{gpu_idx}): {e}')
        return create_silent_audio(filename, 1500)

def recortar_cena(block, filename):
    try:
        trecho = anime_audio_full[int(block['start'] * 1000):int(block['end'] * 1000)]
        # Fade-out de 150ms e fade-in de 50ms para evitar cliques e vazamentos do "ue" original
        if len(trecho) > 200:
            trecho = trecho.fade_in(50).fade_out(150)
        elif len(trecho) > 100:
            trecho = trecho.fade_in(30).fade_out(70)
        trecho.export(filename, format='mp3')
        return len(trecho) / 1000.0
    except:
        return create_silent_audio(filename, int((block['end'] - block['start']) * 1000))

def get_context_window(block_list, current_id, before=4, after=4):
    """
    Retorna uma string com o contexto narrativo:
    - 'before' blocos anteriores (do tipo NARRACAO) mais próximos
    - bloco atual
    - 'after' blocos seguintes (do tipo NARRACAO) mais próximos
    """
    idx = next((i for i, b in enumerate(block_list) if b['id'] == current_id), 0)

    # Coleta anteriores
    prev_texts = []
    i = idx - 1
    while i >= 0 and len(prev_texts) < before:
        if block_list[i].get('tipo') == 'NARRACAO' and block_list[i].get('translated_text'):
            prev_texts.insert(0, block_list[i]['translated_text'])
        i -= 1

    # Coleta posteriores
    next_texts = []
    i = idx + 1
    while i < len(block_list) and len(next_texts) < after:
        if block_list[i].get('tipo') == 'NARRACAO' and block_list[i].get('translated_text'):
            next_texts.append(block_list[i]['translated_text'])
        i += 1

    parts = []
    if prev_texts:
        parts.append("CONTEXTO ANTERIOR: " + " ".join(prev_texts))
    parts.append("TEXTO ATUAL: " + (block_list[idx].get('translated_text') or ''))
    if next_texts:
        parts.append("CONTEXTO POSTERIOR: " + " ".join(next_texts))
    return "\n".join(parts)

MIN_FRAGMENT_CHARS = 30  # bloco com menos disso é candidato a fusão

def fundir_fragmentos_narracao(blocks):
    """
    Funde blocos NARRACAO consecutivos onde o bloco seguinte é um
    fragmento curto que claramente continua o anterior.
    Preserva blocos CENA intactos.
    """
    result = []
    i = 0
    while i < len(blocks):
        blk = dict(blocks[i])
        # Só tenta fundir se for NARRACAO
        if blk.get('tipo') == 'NARRACAO':
            # Enquanto o próximo for NARRACAO e fragmento curto, absorve
            while (i + 1 < len(blocks)
                   and blocks[i+1].get('tipo') == 'NARRACAO'
                   and len(blocks[i+1].get('translated_text', '')) < MIN_FRAGMENT_CHARS):
                next_blk = blocks[i + 1]
                # Funde texto
                blk['translated_text'] = (
                    blk.get('translated_text', '').rstrip()
                    + ' '
                    + next_blk.get('translated_text', '').strip()
                )
                # Estende o intervalo de tempo
                blk['end'] = next_blk['end']
                i += 1
        result.append(blk)
        i += 1
    # Re-numera IDs
    for j, b in enumerate(result):
        b['id'] = j
    return result

# ──────────────────────────────────────────────────────────────────────────────
# FUNÇÃO PRINCIPAL DA FÁBRICA
# ──────────────────────────────────────────────────────────────────────────────
def run_fabrica():
    modo = "Video Completo"
    prefix   = "block_"
    part_str = str(TTS_PART_NUM)
    save_name = f"{BASE_PATH}/ROTEIRO_ADAPTADO_COMPLETO_FINAL_pt{part_str}.json"

    print(f"\n{'='*60}")
    print(f"  INICIANDO: {modo} | Parte {part_str}")
    print(f"{'='*60}")

    # --- Carregar o JSON ja pre-fatiado desta parte ---
    src = f"{BASE_PATH}/roteiro_pt{part_str}.json"
    if not os.path.exists(src):
        print(f"  AVISO: Arquivo fonte nao encontrado ({src}). Pulando {modo}.")
        return

    with open(src, 'r', encoding='utf-8') as f:
        target_list = json.load(f)

    print(f"  {len(target_list)} blocos carregados de roteiro_pt{part_str}.json")
        
    # Aplica fusão de fragmentos na base (garantindo que IDs novos combinem)
    target_list = fundir_fragmentos_narracao(target_list)
    
    def sanitize_blocks(lista):
        lista = sorted(lista, key=lambda b: b['start'])
        out, cursor = [], 0.0
        for blk in lista:
            blk = dict(blk)
            if blk['start'] < cursor:
                blk['start'] = cursor
            if blk['start'] >= blk['end']:
                continue
            out.append(blk)
            cursor = blk['end']
        return out

    # Resolve sobreposições de tempo ANTES de avaliar a velocidade
    target_list = sanitize_blocks(target_list)
    
    # Se for a parte 1 ou modo unico, sintetiza a introdução
    if TASK_KEY == "omni" or TASK_KEY == "omni-tts-pt1":
        intro_json = f"{BASE_PATH}/intro_info.json"
        if os.path.exists(intro_json):
            print("\n[INTRO] Detectado intro_info.json. Sintetizando áudio da introdução...")
            try:
                with open(intro_json, 'r', encoding='utf-8') as f:
                    intro_data = json.load(f)
                intro_text = intro_data.get("intro_text", "")
                if intro_text:
                    intro_filename = os.path.join(TEMP_DIR, "block_intro.mp3")
                    generate_tts(intro_text, intro_filename, gpu_idx=0)
                    print("[INTRO] Áudio da introdução gerado com sucesso!")
            except Exception as e:
                print("[INTRO ERROR] Falha ao sintetizar áudio da introdução:", e)
    
    if os.path.exists(save_name):
        print(f"  Mesclando com cache existente: {save_name}")
        with open(save_name, 'r', encoding='utf-8') as f:
            cache_list = json.load(f)
            
        cache_map = {b['id']: b for b in cache_list}
        
        for i, base_b in enumerate(target_list):
            c_b = cache_map.get(base_b['id'])
            # Só reaproveita se o original não mudou de lugar (checando start)
            if c_b and c_b.get('start') == base_b.get('start'):
                target_list[i]['translated_text'] = c_b.get('translated_text', base_b.get('translated_text'))
                if 'status' in c_b: target_list[i]['status'] = c_b['status']
                if 'current_file' in c_b: target_list[i]['current_file'] = c_b['current_file']
                if 'current_duration' in c_b: target_list[i]['current_duration'] = c_b['current_duration']
                if 'speed_factor' in c_b: target_list[i]['speed_factor'] = c_b['speed_factor']

    def salvar_progresso():
        with _save_lock:
            with open(save_name, 'w', encoding='utf-8') as f:
                json.dump(target_list, f, ensure_ascii=False, indent=4)
            if 'salvar_no_drive' in globals():
                salvar_no_drive(save_name, f'KAGGLE/AUDIO_DUB/{os.path.basename(save_name)}')

    # --- 1. Geração Inicial (V0) — paralela entre GPU0 e GPU1 ---
    pending = [
        (i, b) for i, b in enumerate(target_list)
        if not (b.get('status') == 'GREEN' and os.path.exists(b.get('current_file', '')))
    ]
    print(f"\n  Gerando audios V0: {len(pending)} blocos pendentes | {_num_gpus} GPU(s)")

    def _process_v0(args):
        enum_idx, block = args
        gpu_idx  = enum_idx % _num_gpus
        filename = os.path.join(TEMP_DIR, f"{prefix}{block['id']}_v0.mp3")
        orig_dur = max(0.1, block['end'] - block['start'])
        if block.get('tipo') == 'CENA':
            dur = recortar_cena(block, filename)
            block['current_file']     = filename
            block['current_duration'] = dur
            block['speed_factor']     = 1.0
            block['status']           = 'GREEN'
        else:
            dur   = generate_tts(block.get('translated_text', ''), filename, gpu_idx)
            ratio = dur / orig_dur if dur > 0 else 1.0
            block['current_file']     = filename
            block['current_duration'] = dur
            block['speed_factor']     = ratio
            block['status']           = 'GREEN' if TARGET_MIN_SPEED <= ratio <= TARGET_MAX_SPEED else 'RED'
        salvar_progresso()
    
    if TASK_KEY.startswith("omni-tts-pt"):
        import shutil
        zip_base_name = f"{BASE_PATH}/omni_tts_pt{part_str}"
        print(f"[ZIP] Compactando TEMP_DIR para {zip_base_name}.zip ...")
        shutil.make_archive(zip_base_name, 'zip', TEMP_DIR)
        zip_full_path = zip_base_name + ".zip"
        if 'salvar_no_drive' in globals():
            salvar_no_drive(zip_full_path, f"KAGGLE/AUDIO_DUB/omni_tts_pt{part_str}.zip")

    with ThreadPoolExecutor(max_workers=max(1, _num_gpus)) as ex:
        list(tqdm(ex.map(_process_v0, pending), total=len(pending), desc=f'V0 {modo[:5]}'))

    # --- 2. Loop de Otimização (PARALELIZADO) ---
    def _process_correction(args):
        """Processa um bloco RED: tenta reescrita e gera áudio, usando a GPU especificada."""
        enum_idx, block = args
        gpu_idx = enum_idx % _num_gpus

        original_dur = max(0.1, block['end'] - block['start'])
        if original_dur < 0.8:
            block['status'] = 'WARNING_SHORT'
            salvar_progresso()
    
    if TASK_KEY.startswith("omni-tts-pt"):
        import shutil
        zip_base_name = f"{BASE_PATH}/omni_tts_pt{part_str}"
        print(f"[ZIP] Compactando TEMP_DIR para {zip_base_name}.zip ...")
        shutil.make_archive(zip_base_name, 'zip', TEMP_DIR)
        zip_full_path = zip_base_name + ".zip"
        if 'salvar_no_drive' in globals():
            salvar_no_drive(zip_full_path, f"KAGGLE/AUDIO_DUB/omni_tts_pt{part_str}.zip")
            return

        current_len   = len(block['translated_text'])
        current_speed = max(0.1, block['speed_factor'])
        target_avg    = (TARGET_MIN_SPEED + TARGET_MAX_SPEED) / 2
        ideal_len     = int(current_len * (target_avg / current_speed))
        REAL_MIN      = max(5, int(ideal_len * 0.90))
        REAL_MAX      = int(ideal_len * 1.10)

        # NOVO: contexto expandido (4 anteriores, 4 seguintes)
        contexto = get_context_window(target_list, block['id'], before=4, after=4)
        prompt_min = REAL_MIN
        prompt_max = REAL_MAX
        valid_text_found = False
        candidate  = ''

        print(f"\n  Bloco {block['id']} | Speed: {current_speed:.2f}x | Meta: {REAL_MIN}-{REAL_MAX} chars")

        for attempt in range(1, MAX_TEXT_RETRIES + 1):
            prompt = (
                f"{contexto}\n"
                f"MISSÃO: Reescreva o TEXTO ATUAL EXATAMENTE entre {prompt_min} e {prompt_max} caracteres.\n"
                f"REGRAS:\n"
                f"1. 3ª PESSOA (Ele/Ela). Nunca 1ª pessoa.\n"
                f"2. Vocabulário simples e entendivel.\n"
                f"3. NÃO repita ideias já expressas no CONTEXTO ANTERIOR.\n"
                f"4. NÃO antecipe ideias do CONTEXTO POSTERIOR.\n"
                f"5. Se o CONTEXTO ANTERIOR termina com o mesmo sujeito, inicie com estrutura diferente.\n"
                f"6. Responda APENAS o texto reescrito, sem aspas ou explicacoes."
            )

            try:
                resp = gemini_generate(
                    model_chain=MODELS_REESCRITA,
                    contents=prompt,
                    task_name=f"Reescrita-{block['id']}-T{attempt}"
                )
                candidate = resp.text.strip().replace('"', '')

            except RuntimeError as e:
                if USE_OPENAI_FALLBACK and client_openai and "OPENAI_FALLBACK" in str(e):
                    try:
                        r = client_openai.chat.completions.create(
                            model=OPENAI_FALLBACK_MODEL,
                            messages=[
                                {'role': 'system', 'content': 'Contador restrito de letras.'},
                                {'role': 'user',   'content': prompt}
                            ],
                            temperature=0.7
                        )
                        candidate = r.choices[0].message.content.strip().replace('"', '')
                    except Exception as oe:
                        print(f'  OpenAI fallback falhou: {oe}')
                        break
                else:
                    print(f'  Erro fatal: {e}')
                    break

            except Exception as e:
                print(f'  Erro API: {e}')
                break

            lc = len(candidate)
            if REAL_MIN <= lc <= REAL_MAX:
                print(f"  [OK] T{attempt}: {lc} chars")
                valid_text_found = True
                break
            else:
                print(f"  [FORA] T{attempt}: {lc} chars. Calibrando...")
                if lc > REAL_MAX:
                    exc = lc - REAL_MAX
                    prompt_max = max(10, prompt_max - exc - 5)
                    prompt_min = max(5, prompt_min - exc - 5)
                else:
                    dif = REAL_MIN - lc
                    prompt_min += dif + 5
                    prompt_max += dif + 5

        if not valid_text_found:
            print(f"  Ignorando Bloco {block['id']} — falhou em {MAX_TEXT_RETRIES} tentativas.")
            salvar_progresso()
    
    if TASK_KEY.startswith("omni-tts-pt"):
        import shutil
        zip_base_name = f"{BASE_PATH}/omni_tts_pt{part_str}"
        print(f"[ZIP] Compactando TEMP_DIR para {zip_base_name}.zip ...")
        shutil.make_archive(zip_base_name, 'zip', TEMP_DIR)
        zip_full_path = zip_base_name + ".zip"
        if 'salvar_no_drive' in globals():
            salvar_no_drive(zip_full_path, f"KAGGLE/AUDIO_DUB/omni_tts_pt{part_str}.zip")
            return

        block['translated_text'] = candidate
        new_file = os.path.join(TEMP_DIR, f"{prefix}{block['id']}_v{iteration}.mp3")
        dur   = generate_tts(candidate, new_file, gpu_idx=gpu_idx)
        ratio = dur / original_dur if dur > 0 else 1.0

        block['current_file']     = new_file
        block['current_duration'] = dur
        block['speed_factor']     = ratio
        block['status']           = 'GREEN' if TARGET_MIN_SPEED <= ratio <= TARGET_MAX_SPEED else 'RED'

        print(f"  Speed: {ratio:.2f}x -> {block['status']}")
        salvar_progresso()
    
    if TASK_KEY.startswith("omni-tts-pt"):
        import shutil
        zip_base_name = f"{BASE_PATH}/omni_tts_pt{part_str}"
        print(f"[ZIP] Compactando TEMP_DIR para {zip_base_name}.zip ...")
        shutil.make_archive(zip_base_name, 'zip', TEMP_DIR)
        zip_full_path = zip_base_name + ".zip"
        if 'salvar_no_drive' in globals():
            salvar_no_drive(zip_full_path, f"KAGGLE/AUDIO_DUB/omni_tts_pt{part_str}.zip")

    red_list  = [b for b in target_list if b.get('status') == 'RED' and b.get('tipo') == 'NARRACAO']
    iteration = 0

    while red_list and iteration < MAX_RETRIES:
        iteration += 1
        print(f"\n  Loop Audio [{modo[:5]}] - Iteracao {iteration}/{MAX_RETRIES} | {len(red_list)} blocos fora do limite.")

        correction_tasks = list(enumerate(red_list))

        with ThreadPoolExecutor(max_workers=max(1, _num_gpus)) as ex:
            list(tqdm(ex.map(_process_correction, correction_tasks),
                      total=len(correction_tasks),
                      desc=f'Correção {modo[:5]}'))

        red_list = [b for b in target_list if b.get('status') == 'RED' and b.get('tipo') == 'NARRACAO']

    print(f"\n  FIM [{modo}]: {len(red_list)} blocos irredutíveis restantes.")
    salvar_progresso()
    
    # --- Compactação Final e Upload do Zip da Parte ---
    import shutil
    zip_base_name = f"{BASE_PATH}/omni_tts_pt{part_str}"
    print(f"[ZIP] Compactando TEMP_DIR para {zip_base_name}.zip ...")
    shutil.make_archive(zip_base_name, 'zip', TEMP_DIR)
    zip_full_path = zip_base_name + ".zip"
    if 'salvar_no_drive' in globals():
        salvar_no_drive(zip_full_path, f"KAGGLE/AUDIO_DUB/omni_tts_pt{part_str}.zip")

    if 'report_tts_part_done' in globals():
        report_tts_part_done(part_str, f"TTS Pt{part_str} concluído e zip enviado ao Drive.")

# ──────────────────────────────────────────────────────────────────────────────
# EXECUÇÃO
# ──────────────────────────────────────────────────────────────────────────────
if should_run("tts"):
    run_fabrica()
    print("\nFABRICA CONCLUIDA!")
    cell_end(9, "done", f"TTS Parte {TTS_PART_NUM} concluido")
else:
    print("TTS não é necessário para esta task. Ignorando execução.")
    cell_end(9, "done", "Gerador ignorado")
